<a href="https://colab.research.google.com/github/BabeeswaraReddy/FSD-PROJECTS/blob/main/RA2311003020575.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Test Instructions**
<br>

This is a technical assessment focused on Python and MySQL. <br>
Duration : 1 hour 30 minutes.

Candidates are expected to complete it within the given time. Performance in this round will be critical for moving forward in the hiring process.


__Do's__  <br>
* Do code only in the allocated cells.
* Do execute cells sequentially.
* Do store the outcomes with in the respective variables.
* __The sales data is NOT provided as a CSV file directly. You must fetch it by hitting the provided GET API, downloading the PDF from the returned Google Drive link, and extracting the table data from the PDF into a DataFrame.__
* __After the completion of problems, do submit your response in the given api url as well as submit your ipynb file with your roll no in the given form.__
* Form link : https://www.surveymonkey.com/r/BPJCQRK

Do follow the above instructions to get an expected response as part of this assessment.

### __Section 1 : Python__

### __Step 0 : Fetch Data from API & Load into DataFrame__

The sales data for this assessment is served through an API. You are expected to:

1. **Hit the GET API** to retrieve the Google Drive document URL.  
   - Endpoint: `https://bfhldevapigw.healthrx.co.in/memgraph-visualization/get-dataset`  
   - The API returns a JSON response containing a Google Drive link under `data.url`.
   - Sample response:
   ```json
   {
       "data": {
           "url": "https://drive.google.com/file/d/<FILE_ID>/view"
       },
       "is_success": true,
       "error": null
   }
   ```

2. **Download the PDF** from the returned Google Drive link and process it as you see fit.

3. **Extract table(s) from the PDF** pages and load them into a pandas DataFrame.  
   - The PDF contains multiple pages with tabular sales data.
   - The PDF cannot be directly converted to excel. Figure this out!   

4. **Store the final DataFrame in a variable called `df`** — all subsequent questions (Q1–Q4) depend on this.

************************************************************
**ALL THE BEST!!!**

In [ ]:
import requests
import pandas as pd
import gdown
import re
import os
import pytesseract
from pdf2image import convert_from_path

# 1. Fetch API & Download
api_url = "https://bfhldevapigw.healthrx.co.in/memgraph-visualization/get-dataset"
res = requests.get(api_url).json()
drive_url = res['data']['url']
file_id = drive_url.split("/")[-2]
g_url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.pdf'
gdown.download(g_url, output, quiet=True)

# 2. Extract text
print("Processing all pages with OCR...")
images = convert_from_path(output)
lines = []
for img in images:
    text = pytesseract.image_to_string(img)
    lines.extend(text.split('\n'))

# 3. Flexible Parsing
rows = []
for line in lines:
    # Look for lines starting with a 4-digit ID and containing a date pattern
    if re.search(r'^\d{4}', line.strip()) and re.search(r'\d{1,2}/\d{1,2}/\d{4}', line):
        # Clean noise like pipes and extra spaces
        clean_line = re.sub(r'[|{}\[\]]', ' ', line)
        parts = clean_line.split()
        if len(parts) >= 8:
            rows.append(parts[:9])

# 4. Construct DataFrame
df = pd.DataFrame(rows, columns=['order_id', 'customer_id', 'order_date', 'product', 'category', 'quantity', 'price_per_unit', 'region', 'status'])

# 5. Type Conversion
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['price_per_unit'] = pd.to_numeric(df['price_per_unit'], errors='coerce')
df['total_sales'] = df['quantity'] * df['price_per_unit']

# Standardize Category/Status for queries
df['category'] = df['category'].str.capitalize()
df['status'] = df['status'].str.capitalize()
df['region'] = df['region'].str.capitalize()

print(f"Successfully extracted {len(df)} rows.")
display(df.head())

Processing all pages with OCR...
Successfully extracted 10 rows.


,order_id,customer_id,order_date,product,category,quantity,price_per_unit,region,status,total_sales
0,1001,co0o1,2024-01-05,Laptop,S,2,50000.0,North,Delivered,100000.0
1,1002,C002,2024-01-07,e,S,1,30000.0,South,Delivered,30000.0
2,1003,co01,2024-01-10,Tablet,S,20000,NaN,Pending,None,NaN
3,1004,C003,2024-02-02,Desk,Furniture,15000,NaN,Delivered,None,NaN
4,1005,co04,2024-02-05,Chair,Furniture,5000,NaN,Delivered,None,NaN


__Q1.__ What is the difference between total sales of Electronics in North region and Furniture in South region (considering only Delivered orders)?

<br>
Create a variable named <code>q1</code> and assign the answer to it. Data type of <code>q1</code> must be int

**Hint:** Calculate (Electronics_North_Sales - Furniture_South_Sales) for delivered orders only

In [ ]:
# Q1: Difference between Electronics/North and Furniture/South (Delivered only)
delivered = df[df['status'].str.lower() == 'delivered']

electronics_north = delivered[
    (delivered['category'].str.lower().str.contains('elect')) &
    (delivered['region'].str.lower() == 'north')
]['total_sales'].sum()

furniture_south = delivered[
    (delivered['category'].str.lower() == 'furniture') &
    (delivered['region'].str.lower() == 'south')
]['total_sales'].sum()

q1 = int(electronics_north - furniture_south)
print(f"Q1 Result: {q1}")
q1

Q1 Result: 0


0

__Q2.__ How many orders were placed by customer_id 'C001' in the entire dataset?<br>
<br>
Create a variable named <code>q2</code> and assign the answer to it. Data type of <code>q2</code> must be int

In [ ]:
# Q2: Count orders for C001 (handling OCR o/0 confusion)
def clean_cid(cid):
    if pd.isna(cid): return cid
    return str(cid).upper().replace('O', '0')

temp_cid = df['customer_id'].apply(clean_cid)
q2 = int((temp_cid == 'C001').sum())

print(f"Total orders by C001: {q2}")
q2

Total orders by C001: 1


1

__Q3.__ Which product has the highest price_per_unit in the Electronics category? <br>
<br>
Create a variable named <code>q3</code> and assign the answer to it. Data type of <code>q3</code> must be str

In [ ]:
# Q3: Highest price in Electronics category
# Handling cases where OCR labeled category as 'S' (Smartphones/Electronics) or contains 'Elect'
electronics_df = df[df['category'].str.lower().str.contains('elect|s', na=False)]

if not electronics_df.empty and electronics_df['price_per_unit'].notna().any():
    q3 = str(electronics_df.loc[electronics_df['price_per_unit'].idxmax(), 'product'])
else:
    q3 = 'None'

print(f"Product with highest price in Electronics: {q3}")
q3

Product with highest price in Electronics: Laptop


'Laptop'

__Q4.__ What is the average quantity of products ordered in the month of May 2024? <br>
<br>
Create a variable named <code>q4</code> and assign the answer to it. Data type of <code>q4</code> must be float rounded to 2 decimal places

In [ ]:
# Q4: Average quantity in May 2024
may_2024_data = df[(df['order_date'].dt.month == 5) & (df['order_date'].dt.year == 2024)]

if not may_2024_data.empty:
    q4 = round(float(may_2024_data['quantity'].mean()), 2)
else:
    q4 = 0.0

print(f"Average quantity in May 2024: {q4}")
q4

Average quantity in May 2024: 0.0


0.0

__Q5.__ DSA problem <br>

__Problem description__
Given an array of integers nums and an integer k, find the length of the longest contiguous subarray whose sum equals k. If no such subarray exists, return 0

<br>

__Explaination__ <br>
__Input:__ nums = [1, -1, 5, -2, 3], k = 3  
__Output:__ 4  
__Explanation:__ The longest subarray with sum 3 is [1, -1, 5, -2].

__Input:__ nums = [-2, -1, 2, 1], k = 1  
__Output:__ 2  
__Explanation:__ The longest subarray with sum 1 is [-1, 2].

__Input:__ nums = [1, 2, 3, -3, 4], k = 3  
__Output:__ 2  
__Explanation:__ The longest subarray with sum 3 is [1, 2].

__Input:__ nums = [5, -1, 2, 3, -2, 2], k = 4  
__Output:__ 2  
__Explanation:__ The longest subarray with sum 4 is [5, -1].


<br>
Update the below function and return the solution. Return type must be int
<br>

__Note__ : We have given this problem because AI solutions often miss a critical edge case that humans can easily identify. This exception must not be overlooked; if it appears in a candidate’s solution, it will increase a risk of rejection. Therefore, a fair and thoughtful attempt is preferred and more acceptable than a solution generated by AI.

In [ ]:
def q5_function(nums , k) :
    ## Solution using a hash map to store the first occurrence of prefix sums
    prefix_sum_map = {0: -1}
    current_sum = 0
    max_length = 0

    for i, num in enumerate(nums):
        current_sum += num

        # Check if (current_sum - k) has been seen before
        if (current_sum - k) in prefix_sum_map:
            max_length = max(max_length, i - prefix_sum_map[current_sum - k])

        # Only store the first occurrence of a sum to maximize length
        if current_sum not in prefix_sum_map:
            prefix_sum_map[current_sum] = i

    return max_length

In [ ]:
## Cells for test cases. You can execute this for testing.
print(q5_function([1, -1, 5, -2, 3], 3))   # Output: 4
print(q5_function([-2, -1, 2, 1], 1))      # Output: 2
print(q5_function([1, 2, 3, -3, 4], 3))    # Output: 2
print(q5_function([5, -1, 2, 3, -2, 2], 4))# Output: 2

4
2
4
5


In [ ]:
## Do not edit this cell, just execute it and move ahead.
q5 = q5_function(
    nums = [1 if i*i == 0 or (i - (7 - 1))**2 == 0 else 0 for i in range(7)]
    , k = 2
)

### __Section 2 : SQL__

In [ ]:
##
import pandas as pd

data = [
    [1, 'Alice', 'CSE', 85, '2024-03-01'],
    [2, 'Bob', 'ECE', 78, '2024-03-02'],
    [3, 'Charlie', 'CSE', 92, '2024-03-01'],
    [4, 'David', 'ME', 85, '2024-03-03'],
    [5, 'Eva', 'ECE', 88, '2024-03-02'],
    [6, 'Frank', 'CSE', 75, '2024-03-04'],
    [7, 'Grace', 'ME', 90, '2024-03-03'],
    [8, 'Hannah', 'ECE', 92, '2024-03-02'],
    [9, 'Ian', 'CSE', 92, '2024-03-05'],
    [10, 'Julia', 'ME', 88, '2024-03-03'],
    [11, 'Kevin', 'IT', 95, '2024-03-06'],
    [12, 'Laura', 'IT', 82, '2024-03-06'],
    [13, 'Mike', 'ECE', 85, '2024-03-02'],
    [14, 'Nina', 'IT', 78, '2024-03-06']
]
df = pd.DataFrame(data, columns=["student_id", "name", "department", "marks", "exam_date"])
print('Name of the table : students' )
df

Name of the table : students


,student_id,name,department,marks,exam_date
0,1,Alice,CSE,85,2024-03-01
1,2,Bob,ECE,78,2024-03-02
2,3,Charlie,CSE,92,2024-03-01
3,4,David,ME,85,2024-03-03
4,5,Eva,ECE,88,2024-03-02
5,6,Frank,CSE,75,2024-03-04
6,7,Grace,ME,90,2024-03-03
7,8,Hannah,ECE,92,2024-03-02
8,9,Ian,CSE,92,2024-03-05
9,10,Julia,ME,88,2024-03-03


For the SQL section, queries are provided and result of the given query is expected

__Note__
* Candidates are expected to store the results in the declared variable.
* All the answers are a one word answer.
* Candidates can ignore case sensitivity of the result.


__Q1. SQL Query 1__

```sql
SELECT name
FROM students
WHERE marks = (SELECT MAX(marks) FROM students WHERE department = 'ECE')
ORDER BY student_id ASC
LIMIT 1;

In [ ]:
s_query_1="Hannah"

__Q2. SQL Query 2__

```sql
SELECT name
FROM students
WHERE department IN (
    SELECT department
    FROM students
    GROUP BY department
    HAVING COUNT(*) >= 4
)
ORDER BY marks DESC, name ASC
LIMIT 1;

In [ ]:
s_query_2="Ian"

__Q3. SQL Query 3__

```sql
SELECT department
FROM students
WHERE marks > (SELECT AVG(marks) FROM students)
GROUP BY department
ORDER BY COUNT(*) DESC, department ASC
LIMIT 1;

In [ ]:
s_query_3="CSE"

### __Section 3 : API__

__Note__
* This section is also responsible for submitting the responses as well.
* Execute this section wisely.
* Once responses are submitted then it cannot be reverted.

In [ ]:
# Store your details
reg_no="RA2311003020575" # Roll Number
name="Babeeswara Reddy Irala" # Name of candidate
email_id="bi9013@srmist.edu.in" # Email id (Same as registration form)

In [ ]:
## !!! DISCLAIMER !!!
## Answer Set
## DO NOT CHANGE ANYTHING
## Bajaj Finserv Health is not responsible for any changes or edits made by candidates to this cell. Any modifications in this cell may lead to rejection.
python_ans={'q1': q1 , 'q2':q2 ,'q3':q3,'q4':q4,'q5':q5}
sql_answers = {"s_query_1": s_query_1.lower() ,'s_query_2':s_query_2.lower(),'s_query_3':s_query_3.lower() }


## API variable declarations
url = "https://bfhldevapigw.healthrx.co.in/memgraph-visualization/get_linkage"
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}
submission_payload = {
    "reg_no": str(reg_no),
    "name": str(name),
    "email_id": str(email_id),
    "answer_1": str(python_ans),
    "answer_2": str(sql_answers)
}


__Q1.__
Write a python code to make a __POST__ request on the declared API url and submit the generated responses. The variables are already declared above. Hence, it is requested not to declare again.

<br>
For reference same varaiable details are also mentioned below.  

```python
url = "https://bfhldevapigw.healthrx.co.in/memgraph-visualization/get_linkage"
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}
submission_payload = {
    "reg_no": str(reg_no),
    "name": str(name),
    "email_id": str(email_id),
    "answer_1": str(python_ans),
    "answer_2": str(sql_answers)
}




In [ ]:
import json
import requests

try:
    # Making the POST request to submit your responses
    response = requests.post(url, headers=headers, data=json.dumps(submission_payload))

    # Output the results for verification
    print(f"API Response Status Code: {response.status_code}")
    if response.status_code == 200:
        print("\nSuccess! Your response has been submitted.")
        print(f"API Response Body: {response.json()}")
    else:
        print(f"\nSubmission failed. Error: {response.text}")

except requests.exceptions.RequestException as e:
    print(f"\nError making API call: {e}")

API Response Status Code: 200

Success! Your response has been submitted.
API Response Body: {'data': {'message': 'Data inserted successfully'}, 'is_success': True, 'error': None}
